In [100]:
from __future__ import annotations

from pydantic import BaseModel, Field, ConfigDict
import requests


topicos = [
    'Aposentadoria e Pensão',
		'Categoria Profissional Especial',
		'Contrato Individual de Trabalho',
		'Direito Coletivo do Trabalho',
		'Direito de Greve / Lockout',
		'Direito Individual do Trabalho',
		'Direito Sindical e Questões Análogas',
		'Duração do Trabalho',
		'Férias',
		'Outras Relações de Trabalho',
		'Prescrição',
		'Prescrição e Decadência no Direito do Trabalho',
		'Questões de Alta Complexidade, Grande Impacto e Repercussão',
		'Redução à Condição Análoga à de Escravo',
		'Rescisão do Contrato de Trabalho',
		'Rescisão do Contrato de Trabalho',
		'Responsabilidade Civil do Empregador',
		'Responsabilidade Solidária / Subsidiária',
		'Sentença Normativa',
		'Verbas Remuneratórias, Indenizatórias e Benefícios'
]

query_ = {
  "size": 100,
  "query": {
    "bool": {
      "must": [
        { "match": { "assuntos.nome": nome} } for nome in topicos
      ]
    }
  }
}


query = {
  "size": 10,
  "query": {
    "bool": {
      "must": [
        { "match": { "assuntos.nome": "Transporte Aéreo" } },
        { "match": { "assuntos.nome": "Indenização por Dano Moral" } }
      ]
    }
  }
}

url = "https://api-publica.datajud.cnj.jus.br/api_publica_trt6/_search"

headers = {
    "Authorization": "APIKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="
}

In [101]:
class Classe(BaseModel):
    codigo: int
    nome: str


class Sistema(BaseModel):
    codigo: int
    nome: str


class Formato(BaseModel):
    codigo: int
    nome: str


class ComplementosTabelado(BaseModel):
    codigo: int
    valor: int | None = None
    nome: str
    descricao: str | None = None


class OrgaoJulgador(BaseModel):
    codigo: str
    nome: str


class Movimento(BaseModel):
    codigo: int
    nome: str
    dataHora: str
    orgaoJulgador: OrgaoJulgador | None = None
    
    # complementosTabelados: list[ComplementosTabelado] = []


class OrgaoJulgador1(BaseModel):
    codigoMunicipioIBGE: int
    codigo: int
    nome: str


class Assunto(BaseModel):
    codigo: int
    nome: str


class Source(BaseModel):
    numeroProcesso: str
    classe: Classe
    sistema: Sistema
    formato: Formato
    tribunal: str
    dataHoraUltimaAtualizacao: str
    grau: str
    timestamp: str = Field(..., alias='@timestamp')
    dataAjuizamento: str
    movimentos: list[Movimento]
    id: str
    nivelSigilo: int
    orgaoJulgador: OrgaoJulgador1
    assuntos: list[Assunto]


class SearchHit(BaseModel):
    source: Source = Field(alias="_source")


class SearchHits(BaseModel):
    hits: list[SearchHit]


class SearchResponse(BaseModel):
    hits: SearchHits


# ------------------ RESUMIDO ------------------

class ProcessoResumido(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    numeroProcesso: str | None = None
    classe: str | None = None
    sistema: str | None = None
    formato: str | None = None
    tribunal: str | None = None
    dataHoraUltimaAtualizacao: str | None = None
    grau: str | None = None
    timestamp: str = Field(..., alias='@timestamp')
    dataAjuizamento: str | None = None
    id: str | None = None
    orgaoJulgador: str | None = None
    assuntos: list[str] | None = None
    
    # nivelSigilo: int | None = None
    # movimentos: list[Movimento] | None = None


class ProcessoResumidoResponse(BaseModel):
    processos: list[ProcessoResumido]

def mapear_processos(search_response: SearchResponse) -> ProcessoResumidoResponse:
    processos = []
    for hit in search_response.hits.hits:
        processos.append(
            ProcessoResumido(
                numeroProcesso=hit.source.numeroProcesso,
                classe=hit.source.classe.nome,
                sistema=hit.source.sistema.nome,
                formato=hit.source.formato.nome,
                tribunal=hit.source.tribunal,
                dataHoraUltimaAtualizacao=hit.source.dataHoraUltimaAtualizacao,
                grau=hit.source.grau,
                timestamp=hit.source.timestamp,
                dataAjuizamento=hit.source.dataAjuizamento,
                id=hit.source.id,
                orgaoJulgador=hit.source.orgaoJulgador.nome,
                assuntos=list(map(lambda a: a.nome, hit.source.assuntos))
                # movimentos=hit.source.movimentos,
                # nivelSigilo=hit.source.nivelSigilo,
            )
        )

    return ProcessoResumidoResponse(processos=processos)



In [102]:
import json

def fetch():
    response = requests.get(url, headers=headers, json=query, timeout=30)
    response.raise_for_status()
    return response.json()

def fetch_or_load():
    try:
        return fetch()
    except (requests.RequestException, ValueError) as e:
        print(str(e))
        with open('search_response.json', encoding='utf-8') as f:
            return json.load(f)

response = fetch()
print(len(response))

4


In [103]:

search_response = SearchResponse.model_validate(response)
processos = mapear_processos(search_response)

with open("search_response.json", "w", encoding='utf-8') as f:
    f.write(processos.model_dump_json(indent=2, ensure_ascii=False))
